# UNitary Taxation / Profit Shifting Estimates

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from config import *

pd.set_option('display.max_columns', None)
pd.options.display.float_format = '{:,.2f}'.format

# Output directories
output_base = Path(output_tables)
(output_base / 'method_corrected').mkdir(parents=True, exist_ok=True)
(output_base / 'bilateral').mkdir(parents=True, exist_ok=True)

## 1. Constants and Continent Mapping

In [2]:
CBCR_VARS = ['n_employees', 'unrelated_party_revenues', 'tangible_assets_except_cash',
             'payroll', 'stated_capital', 'total_revenues', 'related_party_revenues',
             'holding_or_managing_ip', 'profit_loss_before_income_tax_corrected']

METADATA_COLS = ['partner_jurisdiction', 'etr_average_corrected', 'cit',
                 'tax_revenue_current_usd', 'gvt_health_expenditure', 'region_tjn',
                 'ukt', 'oecd', 'oecd_oct', 'nld_oct']

partner_info_cols = ['iso_partner'] + METADATA_COLS

CONTINENT_CODES = {
    'E_O': 'Europe', 'E': 'Europe',
    'A_O': 'Americas', 'A': 'Americas',
    'F_O': 'Africa', 'F': 'Africa',
    'S_O': 'Asia_Oceania', 'S': 'Asia_Oceania',
}

# Build continent-to-country mapping from unilateral cross data
cross = pd.read_csv(unilateral_cross_data)
continent_mapping = {
    'Europe': set(cross.loc[cross['region_tjn'] == 'Europe', 'iso3']),
    'Americas': set(cross.loc[cross['region_tjn'].isin(
        ['Latin America', 'Northern America', 'Caribbean/American isl.']), 'iso3']),
    'Africa': set(cross.loc[cross['region_tjn'] == 'Africa', 'iso3']),
    'Asia_Oceania': set(cross.loc[cross['region_tjn'].isin(
        ['Asia', 'Oceania']), 'iso3']),
}

for name, countries in continent_mapping.items():
    print(f'{name}: {len(countries)} countries')

Europe: 53 countries
Americas: 57 countries
Africa: 59 countries
Asia_Oceania: 79 countries


In [5]:
def calculate_misalignment(cbcr_data,
                           formula_vars=['n_employees', 'unrelated_party_revenues',
                                         'tangible_assets_except_cash', 'payroll',
                                         'stated_capital', 'total_revenues',
                                         'related_party_revenues', 'holding_or_managing_ip'],
                           weights=[.5, 0,0,.5, 0, 0, 0, 0],
                           profit_var='profit_loss_before_income_tax_corrected',
                           etr_max=1):
    """SOTJ formula: 50% employees, 50% payroll, ETR < 15%"""
    df = cbcr_data.copy()

    df['profit_var_pos'] = df[profit_var].clip(lower=0)
    df['share_profit'] = df['profit_var_pos'] / df.groupby('iso_parent')['profit_var_pos'].transform('sum')

    actual_weights = []
    actual_variables = []
    for i, var in enumerate(formula_vars):
        if var is not None and weights[i] > 0:
            actual_variables.append(f'share_{var}')
            actual_weights.append(weights[i])
            df.loc[df[var] < 0, var] = 0
            df[f'share_{var}'] = df[var] / df.groupby('iso_parent')[var].transform('sum')

    df['share_economy_partner_of_parent'] = (df.loc[:, actual_variables] * actual_weights).sum(axis=1, min_count=len(actual_weights))
    df.loc[(df['share_economy_partner_of_parent'] == 0) & (df[profit_var] > 0), 'share_economy_partner_of_parent'] = 0.01
    df['share_economy_partner_of_parent'] = df['share_economy_partner_of_parent'] / df.groupby('iso_parent')['share_economy_partner_of_parent'].transform('sum')

    df['theoretical_profit'] = df['share_economy_partner_of_parent'] * df.groupby('iso_parent')[profit_var].transform('sum')
    df['misaligned_profit'] = df[profit_var] - df['theoretical_profit']

    df.loc[(df['misaligned_profit'] > 0) & (df['etr_average_corrected'] > etr_max), 'misaligned_profit'] = 0

    def adjust_misalignment(group):
        total_neg = group.loc[group['misaligned_profit'] < 0, 'misaligned_profit'].sum()
        total_pos = group.loc[group['misaligned_profit'] > 0, 'misaligned_profit'].sum()
        if total_neg != 0:
            factor = -total_pos / total_neg
            group.loc[group['misaligned_profit'] < 0, 'misaligned_profit'] *= factor
        return group

    adjusted_parts = []
    for iso_parent, group in df.groupby('iso_parent'):
        adjusted_parts.append(adjust_misalignment(group.copy()))
    df = pd.concat(adjusted_parts, ignore_index=True)

    return df

In [6]:
def calculate_bilateral_by_parent(misalignment_df, year):
    """Calculate bilateral profit shifting estimates."""
    df = misalignment_df.copy()
    bilateral_rows = []

    for iso_parent in df['iso_parent'].unique():
        parent_data = df[df['iso_parent'] == iso_parent].copy()

        havens = parent_data[parent_data['misaligned_profit'] > 0][['iso_partner', 'misaligned_profit', 'etr_average_corrected']].copy()
        total_shifted = havens['misaligned_profit'].sum()
        if total_shifted <= 0:
            continue
        havens['share_of_shifted'] = havens['misaligned_profit'] / total_shifted
        havens = havens.rename(columns={'iso_partner': 'iso_responsible'})

        sufferers = parent_data[parent_data['misaligned_profit'] < 0][['iso_partner', 'misaligned_profit', 'cit']].copy()
        total_lost = abs(sufferers['misaligned_profit'].sum())
        if total_lost <= 0:
            continue
        sufferers['share_of_loss'] = abs(sufferers['misaligned_profit']) / total_lost
        sufferers = sufferers.rename(columns={'iso_partner': 'iso_affected'})

        parent_tax_loss = (abs(sufferers['misaligned_profit']) * sufferers['cit']).sum()

        for _, haven in havens.iterrows():
            for _, sufferer in sufferers.iterrows():
                bilateral_shifted = haven['share_of_shifted'] * sufferer['share_of_loss'] * total_shifted
                bilateral_tax_loss = haven['share_of_shifted'] * sufferer['share_of_loss'] * parent_tax_loss
                bilateral_rows.append({
                    'year': year,
                    'iso_parent': iso_parent,
                    'iso_responsible': haven['iso_responsible'],
                    'iso_affected': sufferer['iso_affected'],
                    'shifted_profit_musd': bilateral_shifted / 1e6,
                    'tax_loss_musd': bilateral_tax_loss / 1e6,
                })

    return pd.DataFrame(bilateral_rows)


def aggregate_country_results(misalignment_df, unique_partners, year):
    """Aggregate misalignment results by iso_partner."""
    country_results = misalignment_df.groupby('iso_partner').agg(
        negative_misalignment=('misaligned_profit', lambda x: x[x < 0].sum()),
        positive_misalignment=('misaligned_profit', lambda x: x[x > 0].sum()),
        theoretical_profit=('theoretical_profit', 'sum'),
        reported_profit=('profit_loss_before_income_tax_corrected', 'sum')
    ).reset_index()

    country_results['negative_misalignment'] = -country_results['negative_misalignment'] / 1e6
    country_results['positive_misalignment'] = country_results['positive_misalignment'] / 1e6
    country_results['theoretical_profit'] = country_results['theoretical_profit'] / 1e6
    country_results['reported_profit'] = country_results['reported_profit'] / 1e6

    country_results = country_results.merge(unique_partners, on='iso_partner', how='left')

    country_results['tax_revenue_loss'] = country_results['negative_misalignment'] * country_results['cit']
    country_results['tax_revenue_gain'] = country_results['positive_misalignment'] * country_results['etr_average_corrected']

    total_pos = country_results['positive_misalignment'].sum()
    total_loss = country_results['tax_revenue_loss'].sum()

    country_results['tax_revenue_loss_caused_pct_of_total'] = country_results['positive_misalignment'] / total_pos if total_pos > 0 else 0
    country_results['tax_revenue_loss_caused_usd'] = country_results['tax_revenue_loss_caused_pct_of_total'] * total_loss
    country_results['tax_revenue_loss_suffered_pct_of_total'] = country_results['tax_revenue_loss'] / total_loss if total_loss > 0 else 0

    country_results['year'] = year
    return country_results

## 3. Load Data and Define Exclusions

### Bad reporters (2022)

Source: `data/raw/CbcR_reporters_over_time.xlsx` (sheet "2022")

**Only domestic plus continents:** AUT, BHR, FIN, GBR, IRL, KOR, MUS, SWE  
**Only domestic vs rest-of-the-world:** CZE, HUN, MAR, NZL, UKR  
**Partial reporters (few granular partners, heavily aggregated):** CAN\*, COL\*\*, NLD\*

These reporters are excluded from the **share computation** but their data is still
distributed and included in the misalignment calculation.

In [7]:
EXCLUSION_CONDITIONS = [
    ('AUT', 2016, 2022),  # Austria: only continents
    ('BHR', 2022, 2022),  # Bahrain: continents only (new in 2022)
    ('CAN', 2022, 2022),  # Canada: few granular partners, heavily aggregated
    ('COL', 2022, 2022),  # Colombia: too few granular partners
    ('CZE', 2019, 2022),  # Czechia: only domestic vs ROW
    ('FIN', 2016, 2022),  # Finland: only continents
    ('GRC', 2017, 2019),  # Greece
    ('GBR', 2017, 2022),  # UK: only continents
    ('HUN', 2018, 2022),  # Hungary: only domestic vs ROW
    ('IMN', 2017, 2020),  # Isle of Man
    ('IRL', 2016, 2022),  # Ireland: only continents
    ('KOR', 2016, 2022),  # Korea: only continents
    ('MAC', 2019, 2021),  # Macau (improved in 2022)
    ('MAR', 2021, 2022),  # Morocco: only domestic vs ROW
    ('MUS', 2019, 2022),  # Mauritius: continents only
    ('NLD', 2016, 2017),  # Netherlands (2016-2017 only)
    ('NLD', 2022, 2022),  # Netherlands: few granular partners in 2022
    ('NOR', 2016, 2017),  # Norway
    ('NZL', 2018, 2022),  # New Zealand: only domestic vs ROW
    ('POL', 2019, 2021),  # Poland (improved in 2022)
    ('SWE', 2016, 2022),  # Sweden: only continents
    ('UKR', 2022, 2022),  # Ukraine: only WXD (new in 2022)
]

def get_excluded_parents_for_year(year):
    return {iso for iso, start, end in EXCLUSION_CONDITIONS if start <= year <= end}

# Load main dataset
cbcr_full = pd.read_csv(f'{data_final}/cbcr_main_no_imputation_allsubgroupsonly.csv')

print(f'Loaded {len(cbcr_full)} rows')
print(f"Years: {sorted(cbcr_full['year'].unique())}")
print('\nExcluded from share computation per year:')
for year in range(first_year, first_year + n_years):
    excluded = get_excluded_parents_for_year(year)
    print(f'  {year}: {len(excluded)} countries - {sorted(excluded)}')

Loaded 17874 rows
Years: [np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022)]

Excluded from share computation per year:
  2016: 7 countries - ['AUT', 'FIN', 'IRL', 'KOR', 'NLD', 'NOR', 'SWE']
  2017: 10 countries - ['AUT', 'FIN', 'GBR', 'GRC', 'IMN', 'IRL', 'KOR', 'NLD', 'NOR', 'SWE']
  2018: 10 countries - ['AUT', 'FIN', 'GBR', 'GRC', 'HUN', 'IMN', 'IRL', 'KOR', 'NZL', 'SWE']
  2019: 14 countries - ['AUT', 'CZE', 'FIN', 'GBR', 'GRC', 'HUN', 'IMN', 'IRL', 'KOR', 'MAC', 'MUS', 'NZL', 'POL', 'SWE']
  2020: 13 countries - ['AUT', 'CZE', 'FIN', 'GBR', 'HUN', 'IMN', 'IRL', 'KOR', 'MAC', 'MUS', 'NZL', 'POL', 'SWE']
  2021: 13 countries - ['AUT', 'CZE', 'FIN', 'GBR', 'HUN', 'IRL', 'KOR', 'MAC', 'MAR', 'MUS', 'NZL', 'POL', 'SWE']
  2022: 16 countries - ['AUT', 'BHR', 'CAN', 'COL', 'CZE', 'FIN', 'GBR', 'HUN', 'IRL', 'KOR', 'MAR', 'MUS', 'NLD', 'NZL', 'SWE', 'UKR']


## 4. Run Estimation for All Years

In [8]:
def run_estimation(year):
    """Run profit shifting estimation for a given year."""
    print(f'\n{"="*60}\nYear {year}\n{"="*60}')

    cbcr_year = cbcr_full[cbcr_full['year'] == year].copy()
    excluded_parents = get_excluded_parents_for_year(year)

    # Step 1: Compute distribution shares from good reporters
    continent_shares, global_shares = compute_distribution_shares(
        cbcr_year, excluded_parents, continent_mapping
    )

    n_good = cbcr_year[~cbcr_year['iso_parent'].isin(excluded_parents)]['iso_parent'].nunique()
    n_bad = cbcr_year[cbcr_year['iso_parent'].isin(excluded_parents)]['iso_parent'].nunique()
    print(f'  Reporters: {n_good} good + {n_bad} excluded from share computation')

    # Step 2: Build partner metadata lookup
    partner_lookup = (
        cbcr_year[~cbcr_year['iso_partner'].isin(non_countries)]
        .drop_duplicates(subset=['iso_partner'])
        .set_index('iso_partner')[METADATA_COLS]
    )

    # Step 3: Process ALL reporters
    all_processed = []
    total_distributed = 0
    reporters_with_distribution = 0

    for iso_parent in cbcr_year['iso_parent'].unique():
        parent_data = cbcr_year[cbcr_year['iso_parent'] == iso_parent]
        processed, n_dist, n_cont = distribute_aggregates_for_reporter(
            parent_data, iso_parent, continent_mapping,
            continent_shares, global_shares
        )
        all_processed.append(processed)
        total_distributed += n_dist
        if n_dist > 0:
            reporters_with_distribution += 1

    print(f'  Distributed {total_distributed} rows across {reporters_with_distribution} reporters')

    # Step 4: Combine and fill metadata
    combined = pd.concat(all_processed, ignore_index=True)

    for col in METADATA_COLS:
        if col in combined.columns:
            mask = combined[col].isna()
            if mask.any():
                combined.loc[mask, col] = combined.loc[mask, 'iso_partner'].map(partner_lookup[col])
        else:
            combined[col] = combined['iso_partner'].map(partner_lookup[col])

    # Step 5: Run misalignment
    final_misalignment = calculate_misalignment(combined)
    final_misalignment['year'] = year

    available_cols = [c for c in partner_info_cols if c in final_misalignment.columns]
    unique_partners = final_misalignment.drop_duplicates(subset=['iso_partner'])[available_cols]

    country_results = aggregate_country_results(final_misalignment, unique_partners, year)
    bilateral = calculate_bilateral_by_parent(final_misalignment, year)

    total_pos = country_results['positive_misalignment'].sum()
    total_loss = country_results['tax_revenue_loss'].sum()
    print(f'  Shifted: {total_pos:,.0f}M USD, Tax Loss: {total_loss:,.0f}M USD')

    return final_misalignment, country_results, bilateral

In [9]:
results = {'country': [], 'bilateral': [], 'aggregate': []}

for year in range(first_year, first_year + n_years):
    mis, country, bilateral = run_estimation(year)
    if country is not None:
        results['country'].append(country)
        results['bilateral'].append(bilateral)
        total_pos = country['positive_misalignment'].sum()
        total_loss = country['tax_revenue_loss'].sum()
        results['aggregate'].append({
            'year': year,
            'total_shifted_musd': total_pos,
            'total_tax_loss_musd': total_loss
        })
        country.to_csv(output_base / 'method_corrected' / f'country_results_{year}.csv', index=False)
        bilateral.to_csv(output_base / 'method_corrected' / f'bilateral_{year}.csv', index=False)

# Combine all years
if results['country']:
    pd.concat(results['country']).to_csv(output_base / 'method_corrected' / 'country_results_all_years.csv', index=False)
    pd.concat(results['bilateral']).to_csv(output_base / 'method_corrected' / 'bilateral_all_years.csv', index=False)

agg = pd.DataFrame(results['aggregate'])
print('\nAggregate Results (in million USD):')
print(agg.to_string(index=False))


Year 2016
  Reporters: 19 good + 7 excluded from share computation
  Distributed 3391 rows across 26 reporters
  Shifted: 839,226M USD, Tax Loss: 230,251M USD

Year 2017
  Reporters: 28 good + 10 excluded from share computation
  Distributed 5823 rows across 38 reporters
  Shifted: 1,703,442M USD, Tax Loss: 425,199M USD

Year 2018
  Reporters: 36 good + 10 excluded from share computation
  Distributed 6801 rows across 46 reporters
  Shifted: 1,646,177M USD, Tax Loss: 400,597M USD

Year 2019
  Reporters: 36 good + 14 excluded from share computation
  Distributed 7123 rows across 50 reporters
  Shifted: 1,819,916M USD, Tax Loss: 438,624M USD

Year 2020
  Reporters: 39 good + 13 excluded from share computation
  Distributed 7626 rows across 52 reporters
  Shifted: 3,366,698M USD, Tax Loss: 709,330M USD

Year 2021
  Reporters: 41 good + 11 excluded from share computation
  Distributed 7722 rows across 52 reporters
  Shifted: 2,469,970M USD, Tax Loss: 565,688M USD

Year 2022
  Reporters: 3

## 5. Aggregate Bilateral Results

In [10]:
if results['bilateral']:
    bilateral_all = pd.concat(results['bilateral'])

    bilateral_agg = bilateral_all.groupby(['year', 'iso_responsible', 'iso_affected']).agg(
        shifted_profit_musd=('shifted_profit_musd', 'sum'),
        tax_loss_musd=('tax_loss_musd', 'sum'),
        n_reporters=('iso_parent', 'nunique')
    ).reset_index()

    bilateral_agg.to_csv(output_base / 'bilateral' / 'bilateral_aggregated_corrected.csv', index=False)

    tjn_bilateral_path = Path(tjn_shared_bilateral)
    if tjn_bilateral_path.exists():
        bilateral_agg.to_csv(tjn_bilateral_path / 'corporate_taxabuse_iffportal.csv', index=False)
        print(f"Saved: {tjn_bilateral_path / 'corporate_taxabuse_iffportal.csv'}")

    latest_year = bilateral_agg['year'].max()
    latest = bilateral_agg[bilateral_agg['year'] == latest_year]
    print(f'\nTop 15 bilateral pairs by tax loss ({latest_year}):')
    print(latest.nlargest(15, 'tax_loss_musd')[['iso_responsible', 'iso_affected', 'tax_loss_musd']].to_string(index=False))

Saved: C:\Users\aliso\Tax Justice Network Ltd\TJN - Shared Documents\Research team\Projects long-term\SOTJ\Tables\bilateral\corporate_taxabuse_iffportal.csv

Top 15 bilateral pairs by tax loss (2022):
iso_responsible iso_affected  tax_loss_musd
            IRL          USA       7,757.41
            SGP          USA       5,677.04
            CHN          GBR       5,399.98
            SGP          GBR       5,322.76
            BRA          BEL       5,278.50
            CHE          FRA       4,696.29
            IRL          GBR       4,402.80
            IRL          IND       4,144.06
            CHE          USA       4,137.25
            CAN          USA       3,860.85
            JPN          GBR       3,832.47
            IRL          VNM       3,411.33
            SGP          IND       3,078.75
            CHE          GBR       2,949.85
            USA          GBR       2,919.11


In [11]:
# Summary by tax haven (responsible)
if results['bilateral']:
    by_responsible = bilateral_agg.groupby(['year', 'iso_responsible']).agg(
        tax_loss_caused_musd=('tax_loss_musd', 'sum'),
        n_affected=('iso_affected', 'nunique')
    ).reset_index().sort_values(['year', 'tax_loss_caused_musd'], ascending=[True, False])

    by_responsible.to_csv(output_base / 'bilateral' / 'summary_by_responsible_corrected.csv', index=False)

    print(f'\nTop 15 tax havens by harm caused ({latest_year}):')
    print(by_responsible[by_responsible['year'] == latest_year].head(15).to_string(index=False))


Top 15 tax havens by harm caused (2022):
 year iso_responsible  tax_loss_caused_musd  n_affected
 2022             IRL             46,885.16         204
 2022             CHE             36,566.34         204
 2022             CAN             33,026.50         204
 2022             SGP             31,773.08         204
 2022             HKG             21,705.95         204
 2022             CHN             20,140.07         203
 2022             BRA             19,398.38         204
 2022             JPN             14,119.18         203
 2022             LBY             12,047.37         204
 2022             NOR             11,275.25         203
 2022             PRI             11,233.69         204
 2022             USA             10,907.36         198
 2022             MEX             10,152.02         201
 2022             TWN              9,205.55         204
 2022             DEU              8,615.48         199


In [12]:
# Summary by affected country
if results['bilateral']:
    by_affected = bilateral_agg.groupby(['year', 'iso_affected']).agg(
        tax_loss_suffered_musd=('tax_loss_musd', 'sum'),
        n_responsible=('iso_responsible', 'nunique')
    ).reset_index().sort_values(['year', 'tax_loss_suffered_musd'], ascending=[True, False])

    by_affected.to_csv(output_base / 'bilateral' / 'summary_by_affected_corrected.csv', index=False)

    print(f'\nTop 15 countries by harm suffered ({latest_year}):')
    print(by_affected[by_affected['year'] == latest_year].head(15).to_string(index=False))


Top 15 countries by harm suffered (2022):
 year iso_affected  tax_loss_suffered_musd  n_responsible
 2022          GBR               56,055.24            204
 2022          USA               45,539.31            202
 2022          FRA               23,963.98            205
 2022          IND               17,939.39            202
 2022          VNM               15,767.22            202
 2022          DEU               13,774.48            205
 2022          CYM               13,201.47            200
 2022          UKR               11,421.41            199
 2022          HKG               11,214.56            201
 2022          ISR               11,143.63            202
 2022          BEL               10,641.26            205
 2022          CHN               10,458.41            201
 2022          NLD               10,360.40            205
 2022          ESP                9,811.03            205
 2022          BLZ                9,558.43            203


## 6. Summary

In [ ]:
print('\n' + '='*80)
print('SUMMARY')
print('='*80)
print('\nFormula: SOTJ (50% employees, 50% payroll)')
print('ETR threshold: 15%')
print('Method: Unified aggregate distribution (all reporters)')
print('  - _O codes distributed to unreported countries per continent')
print('  - Single-letter continent codes: residual distributed after subtracting reported')
print('  - WXD distributed globally (only for reporters with no continent breakdown)')
print(f"\nFiles saved:")
print(f"  - Country results: {output_base / 'method_corrected'}")
print(f"  - Bilateral results: {output_base / 'bilateral'}")


SUMMARY

Formula: SOTJ (50% employees, 50% payroll)
ETR threshold: 15%
Method: Unified aggregate distribution (all reporters)
  - _O codes distributed to unreported countries per continent
  - Single-letter continent codes: residual distributed after subtracting reported
  - WXD distributed globally (only for reporters with no continent breakdown)

Files saved:
  - Country results: ..\output\tables\UT\SOTJ\method_corrected
  - Bilateral results: ..\output\tables\UT\SOTJ\bilateral


: 